# SHIPP Stage 5B — Silver Requests Development

**Method:** Spark Window Current-State Reconstruction

```text
Bronze request history
    ↓
partition by request_id
    ↓
newest CDC event
    ↓
quality checks
    ↓
silver_requests
```


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SOURCE = "bootcamp_students.shipp_bronze.lb_requests_history"
TARGET = "bootcamp_students.shipp_silver.silver_requests"

bronze = spark.table(SOURCE)
print("Bronze request history rows:", bronze.count())
bronze.printSchema()


In [ ]:
display(
    bronze.groupBy("_pg_change_type")
    .count()
    .orderBy("_pg_change_type")
)


In [ ]:
latest_window = Window.partitionBy("request_id").orderBy(
    F.col("_pg_lsn").desc(),
    F.col("_sort_by").desc(),
)

ranked = bronze.withColumn(
    "_current_rn",
    F.row_number().over(latest_window),
)

current_requests = (
    ranked
    .filter(F.col("_current_rn") == 1)
    .filter(F.col("_pg_change_type").isin("insert", "update_postimage"))
    .filter(F.col("request_id").isNotNull())
    .filter(F.col("latitude").between(-90, 90))
    .filter(F.col("longitude").between(-180, 180))
    .select(
        "request_id",
        "requester_id",
        "request_text",
        "category",
        "status",
        "latitude",
        "longitude",
        "need_by_date",
        "updated_at",
        F.col("_pg_lsn").alias("source_pg_lsn"),
        F.col("_sort_by").alias("source_sort_by"),
        F.col("_pg_change_type").alias("source_change_type"),
        F.current_timestamp().alias("silver_processed_at"),
    )
)

display(current_requests)


In [ ]:
duplicate_keys = (
    current_requests.groupBy("request_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
null_keys = current_requests.filter(F.col("request_id").isNull()).count()
invalid_coords = current_requests.filter(
    ~F.col("latitude").between(-90, 90)
    | ~F.col("longitude").between(-180, 180)
).count()

print("current rows:", current_requests.count())
print("duplicate request_id:", duplicate_keys)
print("null request_id:", null_keys)
print("invalid coordinates:", invalid_coords)

assert duplicate_keys == 0
assert null_keys == 0
assert invalid_coords == 0


In [ ]:
display(
    current_requests.filter(
        F.col("request_id") == "demo-request-001"
    )
)


In [ ]:
(
    current_requests.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET)
)

print("Wrote:", TARGET)
print("Silver request rows:", spark.table(TARGET).count())
